In [1]:
# Imports for DiverseVul loading, cleaning, split, and baselines.
# json is needed because some hash values are too large for pandas.read_json.
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from transformers import pipeline

print("Python:", sys.executable)
print("Torch:", torch.__version__, "| CUDA build:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
if not torch.cuda.is_available():
    raise RuntimeError(
        "This kernel cannot see the GPU. "
        "Use kernel: C:\\Users\\sadma\\anaconda3\\python.exe, then Kernel -> Restart, then re-run all cells."
    )


/Users/martenweijer/Code/cis631-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: /Users/martenweijer/Code/cis631-project/.venv/bin/python
Torch: 2.14.0 | CUDA build: None
Device: mps | no CUDA GPU


In [2]:
# Display settings and dataset paths.
# Paths work whether the notebook is run from src/ or the project root.
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 80)

ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
DATA_DIR = ROOT / "Data"

DIVERSVUL_PATH = DATA_DIR / "DiverseVul" / "diversevul_20230702.json"
BIGVUL_PATH = DATA_DIR / "BigVul" / "MSR_data_cleaned.csv"
MEGAVUL_PATH = DATA_DIR / "MegaVul" / "megavul_simple.json"

print("DiverseVul exists:", DIVERSVUL_PATH.exists())

DiverseVul exists: True


## DiverseVul — load and EDA

Primary dataset for RQ1–RQ2. Task: predict **CWE type** from function source code.
This section only inspects the raw file. Cleaning starts later.

In [3]:
diversevul = pd.DataFrame(
    json.loads(line) for line in open(DIVERSVUL_PATH, encoding="utf-8") if line.strip()
)
print("Raw shape:", diversevul.shape)
diversevul.head()

Raw shape: (330492, 8)


,func,target,cwe,project,commit_id,hash,size,message
0,"int _gnutls_ciphertext2compressed(gnutls_session_t session,\n\t\t\t\t opaqu...",1,[],gnutls,7ad6162573ba79a4392c63b453ad0220ca6c5ace,73008646937836648589283922871188272089,157,added an extra check while checking the padding.
1,static char *make_filename_safe(const char *filename TSRMLS_DC)\n{\n\tif (*f...,1,[CWE-264],php-src,055ecbc62878e86287d742c7246c21606cee8183,211824207069112513181516095447837228041,22,Improve check for :memory: pseudo-filename in SQlite
2,"unpack_Z_stream(int fd_in, int fd_out)\n{\n\tIF_DESKTOP(long long total_writ...",1,[],busybox,251fc70e9722f931eec23a34030d05ba5f747b0e,21401706257394042943815500829552774160,232,uncompress: fix buffer underrun by corrupted input\n\nSigned-off-by: Denys V...
3,"static void cirrus_do_copy(CirrusVGAState *s, int dst, int src, int w, int h...",1,[CWE-787],qemu,b2eb849d4b1fdb6f35d5c46958c7f703cf64cfef,135590882627853658533498335902319684573,66,"CVE-2007-1320 - Cirrus LGD-54XX ""bitblt"" heap overflow\n\nI have just notice..."
4,"glue(cirrus_bitblt_rop_fwd_, ROP_NAME)(CirrusVGAState *s,\n ...",1,[CWE-787],qemu,b2eb849d4b1fdb6f35d5c46958c7f703cf64cfef,27696392987383562433164405181263025184,18,"CVE-2007-1320 - Cirrus LGD-54XX ""bitblt"" heap overflow\n\nI have just notice..."


In [4]:
diversevul.info()
print(diversevul.isna().sum())
print("\nVulnerable flag (target):")
print(diversevul["target"].value_counts(dropna=False))
print("\nEmpty CWE lists:", (diversevul["cwe"].map(len) == 0).sum())
print("Empty func:", (diversevul["func"].astype(str).str.strip() == "").sum())

<class 'pandas.DataFrame'>
RangeIndex: 330492 entries, 0 to 330491
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   func       330492 non-null  str   
 1   target     330492 non-null  int64 
 2   cwe        330492 non-null  object
 3   project    330492 non-null  str   
 4   commit_id  330492 non-null  str   
 5   hash       330492 non-null  object
 6   size       330492 non-null  int64 
 7   message    330492 non-null  str   
dtypes: int64(2), object(2), str(4)
memory usage: 642.4+ MB
func         0
target       0
cwe          0
project      0
commit_id    0
hash         0
size         0
message      0
dtype: int64

Vulnerable flag (target):
target
0    311547
1     18945
Name: count, dtype: int64

Empty CWE lists: 48758
Empty func: 1


- About **330,492** No NA in Pandas
- **48,758** rows have `cwe == []` (which is considered empty).
- **1** row with no code. (Should be dropped)
- `target`: **18,945** vulnerable / **311,547** not vulnerable.
- For CWE classification we do **not** keep all 18,945 vulnerable rows as labeled data (some have no CWE).

In [5]:
print(
    "Vulnerable + empty CWE:",
    ((diversevul["target"] == 1) & (diversevul["cwe"].map(len) == 0)).sum(),
)
print(
    "Not vulnerable + empty CWE:",
    ((diversevul["target"] == 0) & (diversevul["cwe"].map(len) == 0)).sum(),
)
print("Rows with more than one CWE:", (diversevul["cwe"].map(len) > 1).sum())
print("Duplicate func text:", diversevul["func"].duplicated().sum())
print("\nCWE list length counts:")
print(diversevul["cwe"].map(len).value_counts())
print("\nExploded CWE counts (all rows with a CWE; includes target==0):")
print(diversevul.loc[diversevul["cwe"].map(len) > 0, "cwe"].explode().value_counts())
print("\nUnique projects:", diversevul["project"].nunique())

Vulnerable + empty CWE: 2836
Not vulnerable + empty CWE: 45922
Rows with more than one CWE: 71041
Duplicate func text: 0

CWE list length counts:
cwe
1    210693
2     63854
0     48758
3      6801
4       386
Name: count, dtype: int64

Exploded CWE counts (all rows with a CWE; includes target==0):
cwe
CWE-787    39343
CWE-125    28180
CWE-703    25208
CWE-119    25036
CWE-20     22870
           ...  
CWE-672       11
CWE-91         9
CWE-565        8
CWE-825        8
CWE-805        2
Name: count, Length: 150, dtype: int64

Unique projects: 800


- **2,836** vulnerable rows have no CWE (Ignored in supervised learning).
- Labeled vulnerable set size: **18,945 − 2,836 = 16,109**.
- **71,041** rows in the full file have multiple CWEs (many are `target==0`).
- **0** exact duplicate `func` strings.
- **800** projects → enough for a project-wise split later.
- Exploded CWE counts above mix vulnerable and non-vulnerable rows; do not use them as the final rare/common definition.

In [6]:
# Crosstab: vulnerable flag vs empty CWE list.
# False = has at least one CWE; True = empty list.
pd.crosstab(diversevul["target"], diversevul["cwe"].map(len) == 0)

cwe,False,True
target,,
0,265625,45922
1,16109,2836


**EDA note (crosstab)**

| target | has CWE | empty CWE |
|---|---|---|
| 0 (not vul) | 265,625 | 45,922 |
| 1 (vul) | **16,109** | 2,836 |

Many `target==0` rows still carry a CWE from the related CVE/commit. That CWE is **not** a safe training label for "this function's bug type." For the supervised baseline we keep only the **16,109** vul + CWE rows.

## Cleaning for supervised / baseline learning

Goal: one clean table for **code → CWE** supervised learning.

**Rules**
1. Drop empty `func`.
2. Keep `target == 1` and non-empty `cwe` for the labeled set.
3. Assign **one** `cwe_label` per row (rare-aware: keep the rarest CWE in the list).
4. Drop classes with fewer than `MIN_COUNT` examples (default **2**).
5. Keep `func`, `cwe_label`, `project`, `commit_id` for now.

**Why not only `func` + `cwe_label` yet?**  
BigVul/MegaVul notebooks slim to two columns for a quick toy model. We still need `project` for a project-wise split and `commit_id` for later overlap checks (RQ3). Those columns are **not** model features. A separate `model_df` with only `func` and `cwe_label` is built at the end for the baseline fit.

In [7]:
# Minimum examples per CWE after labeling.
# Use 2 so ultra-rare singletons do not break metrics, but RQ2 rare classes mostly remain.
MIN_COUNT = 2

# Work on a copy so the raw EDA frame stays unchanged.
df = diversevul.copy()
print("Start rows:", len(df))

Start rows: 330492


In [8]:
# drop rows with source code.
before = len(df)
df = df[df["func"].astype(str).str.strip() != ""].copy()
print(f"After dropping empty func: {before} -> {len(df)} (dropped {before - len(df)})")

After dropping empty func: 330492 -> 330491 (dropped 1)


In [9]:
# Step 2: build the labeled set for supervised / baseline learning.
# Drop target==0 and rows with no CWE from the labeled table.
labeled = df[(df["target"] == 1) & (df["cwe"].map(len) > 0)].copy()

# Optional pools for SSL later (not used by the supervised baseline).
unlabeled_vul_no_cwe = df[(df["target"] == 1) & (df["cwe"].map(len) == 0)].copy()
unlabeled_non_vul = df[df["target"] == 0].copy()

print("Labeled (vul + CWE):", len(labeled))
print("Unlabeled pool candidate (vul, no CWE):", len(unlabeled_vul_no_cwe))
print("Unlabeled pool candidate (not vul):", len(unlabeled_non_vul))

Labeled (vul + CWE): 16109
Unlabeled pool candidate (vul, no CWE): 2836
Unlabeled pool candidate (not vul): 311546


- Labeled set ≈ **16,109** rows.
- **2,836** vul rows with no CWE are held out of supervised Y; they can join the SSL unlabeled pool later.

In [10]:
# Step 3: rare-aware single CWE label.
# Frequencies are counted on the labeled set only (explode for counts, not for training rows).
cwe_freq = labeled["cwe"].explode().value_counts()
print("Unique CWE ids in labeled lists:", len(cwe_freq))


def pick_cwe_label(cwe_list):
    """Return one CWE id. Prefer the rarest id when the list has several."""
    if not isinstance(cwe_list, list) or len(cwe_list) == 0:
        return None
    if len(cwe_list) == 1:
        return cwe_list[0]
    # Sort by frequency (ascending), then by original position for ties.
    return sorted(
        cwe_list,
        key=lambda cwe: (int(cwe_freq.get(cwe, 0)), cwe_list.index(cwe)),
    )[0]


labeled["cwe_label"] = labeled["cwe"].map(pick_cwe_label)
labeled["cwe_first"] = labeled["cwe"].str[0]

n_multi = int((labeled["cwe"].map(len) > 1).sum())
n_rare_choice = int((labeled["cwe_label"] != labeled["cwe_first"]).sum())
print("Multi-CWE rows in labeled set:", n_multi)
print("Rows where rare-aware label != first CWE:", n_rare_choice)
print("\nTop cwe_label counts:")
print(labeled["cwe_label"].value_counts().head(10))
print("\nBottom cwe_label counts:")
print(labeled["cwe_label"].value_counts().tail(10))

Unique CWE ids in labeled lists: 150
Multi-CWE rows in labeled set: 4215
Rows where rare-aware label != first CWE: 1876

Top cwe_label counts:
cwe_label
CWE-125    1572
CWE-787    1379
CWE-119    1277
CWE-20      958
CWE-476     878
CWE-416     878
CWE-190     716
CWE-200     591
CWE-703     511
CWE-189     405
Name: count, dtype: int64

Bottom cwe_label counts:
cwe_label
CWE-216    1
CWE-349    1
CWE-565    1
CWE-697    1
CWE-798    1
CWE-457    1
CWE-786    1
CWE-805    1
CWE-428    1
CWE-335    1
Name: count, dtype: int64


**Cleaning note (step 3)**

- About **4,215** labeled rows have more than one CWE.
- About **1,876** of those get a label that is **not** the first CWE → the rare-aware rule is changing real decisions.
- We do **not** duplicate rows (same `func` with two labels would break single-label training).

In [11]:
label_counts = labeled["cwe_label"].value_counts()
keep_labels = label_counts[label_counts >= MIN_COUNT].index

before_rows = len(labeled)
before_classes = labeled["cwe_label"].nunique()
labeled = labeled[labeled["cwe_label"].isin(keep_labels)].copy()

print(f"MIN_COUNT = {MIN_COUNT}")
print(f"Rows: {before_rows} -> {len(labeled)} (dropped {before_rows - len(labeled)})")
print(
    f"Classes: {before_classes} -> {labeled['cwe_label'].nunique()} "
    f"(dropped {before_classes - labeled['cwe_label'].nunique()})"
)
print("\nDropped singleton / below-min classes:")
print(label_counts[label_counts < MIN_COUNT].sort_values())

MIN_COUNT = 2
Rows: 16109 -> 16098 (dropped 11)
Classes: 150 -> 139 (dropped 11)

Dropped singleton / below-min classes:
cwe_label
CWE-943    1
CWE-216    1
CWE-349    1
CWE-565    1
CWE-697    1
CWE-798    1
CWE-457    1
CWE-786    1
CWE-805    1
CWE-428    1
CWE-335    1
Name: count, dtype: int64



- With `MIN_COUNT = 2`: **16,098** rows and **139** CWE classes remain (**150 → 139**).
- **11** singleton classes dropped (1 example each): CWE-216, 335, 349, 428, 457, 565, 697, 786, 798, 805, 943.
- Small cut only; most rare classes for RQ2 stay. Rows go **16,109 → 16,098**.

In [12]:
# Step 5: column selection.
# Keep helper columns for split / leakage. Drop text that must not be a model feature.
keep_cols = ["func", "cwe_label", "project", "commit_id"]
labeled_clean = labeled[keep_cols].copy()

# Modeling frame for supervised baseline (code only -> CWE).
# Matches the spirit of big-vul / mega-vul slim tables, without dropping split keys yet.
model_df = labeled_clean[["func", "cwe_label"]].copy()

print("labeled_clean columns:", list(labeled_clean.columns))
print("labeled_clean shape:", labeled_clean.shape)
print("model_df columns:", list(model_df.columns))
print("model_df shape:", model_df.shape)
print("\nUnique projects in cleaned labeled set:", labeled_clean["project"].nunique())
print("Empty func in cleaned set:", (labeled_clean["func"].astype(str).str.strip() == "").sum())
labeled_clean.head()

labeled_clean columns: ['func', 'cwe_label', 'project', 'commit_id']
labeled_clean shape: (16098, 4)
model_df columns: ['func', 'cwe_label']
model_df shape: (16098, 2)

Unique projects in cleaned labeled set: 739
Empty func in cleaned set: 0


,func,cwe_label,project,commit_id
1,static char *make_filename_safe(const char *filename TSRMLS_DC)\n{\n\tif (*f...,CWE-264,php-src,055ecbc62878e86287d742c7246c21606cee8183
3,"static void cirrus_do_copy(CirrusVGAState *s, int dst, int src, int w, int h...",CWE-787,qemu,b2eb849d4b1fdb6f35d5c46958c7f703cf64cfef
4,"glue(cirrus_bitblt_rop_fwd_, ROP_NAME)(CirrusVGAState *s,\n ...",CWE-787,qemu,b2eb849d4b1fdb6f35d5c46958c7f703cf64cfef
5,static int cirrus_bitblt_videotovideo_copy(CirrusVGAState * s)\n{\n if (s...,CWE-787,qemu,b2eb849d4b1fdb6f35d5c46958c7f703cf64cfef
6,"static void cirrus_mem_writeb_mode4and5_8bpp(CirrusVGAState * s,\n\t\t\t\t\t...",CWE-787,qemu,b2eb849d4b1fdb6f35d5c46958c7f703cf64cfef



| Frame | Columns | Use |
|---|---|---|
| `labeled_clean` | `func`, `cwe_label`, `project`, `commit_id` | Cleaning result + future project-wise split |
| `model_df` | `func`, `cwe_label` | Supervised baseline input (code only) |

Dropped on purpose: `message` (not code-only), `hash`, `size`, raw `cwe`, `target` (already filtered).

We **agree** with matching BigVul/MegaVul for the **model** frame (`func` + label). We **do not** delete `project` / `commit_id` from the cleaned training table yet, because the next step is a project-wise split, not a random row split like the current BigVul toy baseline.

In [13]:
# Final cleaning summary for the presentation / next baseline step.
print("=== DiverseVul cleaning summary (supervised / baseline) ===")
print(f"Raw rows:              {len(diversevul):,}")
print(f"Labeled cleaned rows:  {len(labeled_clean):,}")
print(f"CWE classes:           {labeled_clean['cwe_label'].nunique()}")
print(f"Projects:              {labeled_clean['project'].nunique()}")
print(f"MIN_COUNT:             {MIN_COUNT}")
print(f"model_df ready:        {model_df.shape}")
print("\nNext cells: project-wise split, then CodeBERT embeddings + LogReg.")


=== DiverseVul cleaning summary (supervised / baseline) ===
Raw rows:              330,492
Labeled cleaned rows:  16,098
CWE classes:           139
Projects:              739
MIN_COUNT:             2
model_df ready:        (16098, 2)

Next cells: project-wise split, then CodeBERT embeddings + LogReg.


## Project-wise split

Split by **project**, not by random rows. No project appears in more than one of train / val / test.
Use `labeled_clean` (needs `project`). Model features stay code-only (`func` -> `cwe_label`).

Ratios (by project count): train 70%, val 15%, test 15%.


In [14]:
# Project-wise train / val / test split on the cleaned labeled table.
SEED = 42
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15  # remaining projects go to test

projects = labeled_clean["project"].drop_duplicates().to_numpy()
rng = np.random.RandomState(SEED)
projects = projects.copy()
rng.shuffle(projects)

n = len(projects)
n_train = int(TRAIN_FRAC * n)
n_val = int(VAL_FRAC * n)

train_projects = set(projects[:n_train])
val_projects = set(projects[n_train:n_train + n_val])
test_projects = set(projects[n_train + n_val:])

train_df = labeled_clean[labeled_clean["project"].isin(train_projects)].copy()
val_df = labeled_clean[labeled_clean["project"].isin(val_projects)].copy()
test_df = labeled_clean[labeled_clean["project"].isin(test_projects)].copy()

# Sanity: no project leakage across splits.
assert train_projects.isdisjoint(val_projects)
assert train_projects.isdisjoint(test_projects)
assert val_projects.isdisjoint(test_projects)
assert len(train_df) + len(val_df) + len(test_df) == len(labeled_clean)

split_summary = pd.DataFrame([
    {"split": "train", "projects": len(train_projects), "rows": len(train_df), "classes": train_df["cwe_label"].nunique()},
    {"split": "val", "projects": len(val_projects), "rows": len(val_df), "classes": val_df["cwe_label"].nunique()},
    {"split": "test", "projects": len(test_projects), "rows": len(test_df), "classes": test_df["cwe_label"].nunique()},
])
print(split_summary.to_string(index=False))
print("\nClasses in test but not in train:", len(set(test_df["cwe_label"]) - set(train_df["cwe_label"])))
print("Classes in val but not in train:", len(set(val_df["cwe_label"]) - set(train_df["cwe_label"])))


split  projects  rows  classes
train       517 12222      134
  val       110  2089       82
 test       112  1787       76

Classes in test but not in train: 3
Classes in val but not in train: 3


In [16]:
df_clean = labeled_clean[["func", "cwe_label"]].copy()
df_clean = df_clean.rename(columns={"cwe_label": "cwe"})

codebert = pipeline("feature-extraction", model="microsoft/codebert-base")
# Convert each function into one CodeBERT embedding
def embed(code):
    result = codebert(code, truncation=True, max_length=512)
    return np.array(result[0][0])  # first token embedding: 768 numbers

X = np.vstack([embed(code) for code in df_clean["func"]])
y = df_clean["cwe"]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 34711.24it/s]


In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

classifier = LogisticRegression(max_iter=1000)
classifier.fit(X_train, y_train)

predictions = classifier.predict(X_test)
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

    CWE-1021       0.00      0.00      0.00         1
     CWE-113       0.00      0.00      0.00         1
     CWE-119       0.26      0.41      0.32       246
     CWE-120       0.50      0.11      0.19        61
     CWE-121       0.00      0.00      0.00         1
     CWE-122       0.38      0.14      0.21        21
     CWE-125       0.34      0.52      0.41       314
     CWE-126       0.00      0.00      0.00         1
     CWE-129       0.00      0.00      0.00         2
     CWE-131       1.00      0.50      0.67         2
     CWE-134       1.00      0.22      0.36         9
      CWE-16       0.00      0.00      0.00         1
      CWE-17       0.00      0.00      0.00         6
      CWE-18       1.00      1.00      1.00         6
     CWE-189       0.37      0.30      0.33        76
      CWE-19       0.80      0.24      0.36        17
     CWE-190       0.36      0.34      0.35       151
     CWE-191       1.00    

/Users/martenweijer/Code/cis631-project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/martenweijer/Code/cis631-project/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/martenweijer/Co

**Split note**

- Rows will not be exactly 70/15/15 because projects have different sizes.
- Some rare CWEs may appear only in val/test; the baseline still runs, but those classes cannot be learned from train.
- For this supervised baseline we train on **full** `train_df`. For RQ1 later we will carve a small `D_L` from train only.


## Supervised learning (code only)

CodeBERT embeddings + Logistic Regression on `func` -> `cwe_label` only.

Main metric: **macro-F1**. Also report weighted-F1.


In [ ]:
# CodeBERT embeddings (frozen), then Logistic Regression. Requires GPU.
print("Python:", sys.executable)
print("Torch:", torch.__version__, "| CUDA build:", torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not visible. Select interpreter C:\\Users\\sadma\\anaconda3\\python.exe, "
        "then Kernel -> Restart Kernel, then re-run from the imports cell."
    )

print("CodeBERT device: cuda:0 |", torch.cuda.get_device_name(0))
pipe = pipeline(
    "feature-extraction",
    model="microsoft/codebert-base",
    truncation=True,
    device=0,
)
train_df["embeddings"] = train_df["func"].astype(str).apply(lambda x: pipe(x)[0][0])
val_df["embeddings"] = val_df["func"].astype(str).apply(lambda x: pipe(x)[0][0])
test_df["embeddings"] = test_df["func"].astype(str).apply(lambda x: pipe(x)[0][0])

SEED = 42
X_train = np.vstack(train_df["embeddings"].to_list())
X_val = np.vstack(val_df["embeddings"].to_list())
X_test = np.vstack(test_df["embeddings"].to_list())
y_train = train_df["cwe_label"].astype(str)
y_val = val_df["cwe_label"].astype(str)
y_test = test_df["cwe_label"].astype(str)

clf = LogisticRegression(
    max_iter=1000, class_weight="balanced", solver="lbfgs", random_state=SEED
)
clf.fit(X_train, y_train)

y_val_pred = clf.predict(X_val)
y_test_pred = clf.predict(X_test)

metrics = pd.DataFrame([
    {"split": "val", "macro_f1": f1_score(y_val, y_val_pred, average="macro", zero_division=0),
     "weighted_f1": f1_score(y_val, y_val_pred, average="weighted", zero_division=0)},
    {"split": "test", "macro_f1": f1_score(y_test, y_test_pred, average="macro", zero_division=0),
     "weighted_f1": f1_score(y_test, y_test_pred, average="weighted", zero_division=0)},
])
print(metrics.to_string(index=False))
print(classification_report(y_test, y_test_pred, labels=sorted(y_train.unique()), zero_division=0, digits=3))


**Note**

| Split | Projects | Rows | Classes |
|---|---:|---:|---:|
| train | 517 | 12,222 | 134 |
| val | 110 | 2,089 | 82 |
| test | 112 | 1,787 | 76 |

- Frozen CodeBERT (`feature-extraction` -> `[CLS]`) + LogReg. Re-run the cell above for metrics.
- Embedding is slow on CPU because each function is passed through CodeBERT once.
